In [1]:
# ==================================================================
# ЗАДАНИЯ ДЛЯ ПРАКТИЧЕСКОЙ РАБОТЫ К ГЛАВЕ 3
# Вариант 7 (табл. 3.4)
# ==================================================================
# Задание 1 — метод Гаусса (схема единственного деления):
#             а) вручную, три знака после запятой, невязки, определитель
#             б) программой для ЭВМ
# Задание 2 — метод простой итерации, ε = 1e-4:
#             а) с оценкой погрешности по формуле (3.15)
#             б) по эмпирическому критерию близости соседних приближений
# Задание 3 — инструментальное средство, сопоставление результатов

import numpy as np

A = [[0.20,  0.44,  0.81],
     [0.58, -0.29,  0.05],
     [0.05,  0.34,  0.10]]
B = [0.74, 0.02, 0.32]

RESULTS = {}
r3 = lambda v: round(v, 3)          # округление «на калькуляторе»


def show_system(A, b, title=""):
    if title:
        print(title)
    for i in range(3):
        print(f"   {A[i][0]:+.2f}·x₁ {A[i][1]:+.2f}·x₂ {A[i][2]:+.2f}·x₃ = {b[i]:+.2f}")


def residuals(A, b, x):
    return [sum(A[i][j] * x[j] for j in range(3)) - b[i] for i in range(3)]


show_system(A, B, "Вариант 7, система (3.17):")
det_exact = np.linalg.det(np.array(A))
x_exact = np.linalg.solve(np.array(A), np.array(B))
print(f"\n   определитель ≠ 0 ({det_exact:.6f}) — решение существует и единственно")
print(f"   число обусловленности cond(A) = {np.linalg.cond(np.array(A)):.3f}"
      "  (умеренное, система устойчива)")
RESULTS["эталон"] = x_exact


Вариант 7, система (3.17):
   +0.20·x₁ +0.44·x₂ +0.81·x₃ = +0.74
   +0.58·x₁ -0.29·x₂ +0.05·x₃ = +0.02
   +0.05·x₁ +0.34·x₂ +0.10·x₃ = +0.32

   определитель ≠ 0 (0.137857) — решение существует и единственно
   число обусловленности cond(A) = 4.595  (умеренное, система устойчива)


In [2]:
# ===== ЗАДАНИЕ 1а — МЕТОД ГАУССА ВРУЧНУЮ (три знака после запятой) =====
print("=" * 78)
print("ЗАДАНИЕ 1а   Схема единственного деления, расчёт с тремя знаками")
print("=" * 78)
print("""
Схема единственного деления: ведущая строка делится на диагональный
элемент, затем из нижних строк вычитается ведущая, умноженная на их
первый коэффициент. Все промежуточные числа округляются до 0,001 —
именно так, как это получилось бы на калькуляторе.
""")

M = [row[:] + [B[i]] for i, row in enumerate(A)]
lead = []

print("ПРЯМОЙ ХОД")
for k in range(3):
    p = M[k][k]
    lead.append(p)
    print(f"\n  шаг {k + 1}:  ведущий элемент a{k + 1}{k + 1} = {p:+.3f}")
    M[k] = [r3(v / p) for v in M[k]]
    print(f"     ведущая строка / {p:+.3f}:  " +
          "  ".join(f"{v:+.3f}" for v in M[k]))
    for i in range(k + 1, 3):
        m = M[i][k]
        M[i] = [r3(M[i][j] - m * M[k][j]) for j in range(4)]
        print(f"     строка {i + 1} − ({m:+.3f})·ведущая:  " +
              "  ".join(f"{v:+.3f}" for v in M[i]))

print("\n  Треугольный вид:")
for i in range(3):
    print("     " + "  ".join(f"{v:+.3f}" for v in M[i]))

print("\nОБРАТНЫЙ ХОД")
xh = [0.0] * 3
for i in (2, 1, 0):
    s = sum(M[i][j] * xh[j] for j in range(i + 1, 3))
    xh[i] = r3(M[i][3] - s)
    print(f"  x{i + 1} = {M[i][3]:+.3f} − ({s:+.3f}) = {xh[i]:+.3f}")

print(f"\n   РЕШЕНИЕ ВРУЧНУЮ:  x₁ = {xh[0]:.3f},  x₂ = {xh[1]:.3f},  x₃ = {xh[2]:.3f}")

print("\n" + "-" * 78)
print("ПОДСТАНОВКА В ИСХОДНУЮ СИСТЕМУ И НЕВЯЗКИ")
print("-" * 78)
res = residuals(A, B, xh)
for i in range(3):
    lhs = sum(A[i][j] * xh[j] for j in range(3))
    print(f"   уравнение {i + 1}:  получено {lhs:+.6f},  должно быть {B[i]:+.2f},"
          f"   невязка r{i + 1} = {res[i]:+.6f}")
print(f"\n   максимальная невязка: {max(abs(v) for v in res):.2e}")

print("\n" + "-" * 78)
print("ОПРЕДЕЛИТЕЛЬ ЧЕРЕЗ ВЕДУЩИЕ ЭЛЕМЕНТЫ")
print("-" * 78)
det_h = 1.0
for p in lead:
    det_h *= p
print(f"\n   ведущие элементы: " + " · ".join(f"({p:+.3f})" for p in lead))
print(f"   det A = {det_h:+.6f}")
print(f"   для сравнения, точное значение: {det_exact:+.6f}")
print(f"   расхождение {abs(det_h - det_exact):.2e} — накопленная ошибка округлений")
print("""
   Перестановок строк не делалось, поэтому знак определителя менять не нужно.
""")
RESULTS["Гаусс вручную"] = xh


ЗАДАНИЕ 1а   Схема единственного деления, расчёт с тремя знаками

Схема единственного деления: ведущая строка делится на диагональный
элемент, затем из нижних строк вычитается ведущая, умноженная на их
первый коэффициент. Все промежуточные числа округляются до 0,001 —
именно так, как это получилось бы на калькуляторе.

ПРЯМОЙ ХОД

  шаг 1:  ведущий элемент a11 = +0.200
     ведущая строка / +0.200:  +1.000  +2.200  +4.050  +3.700
     строка 2 − (+0.580)·ведущая:  +0.000  -1.566  -2.299  -2.126
     строка 3 − (+0.050)·ведущая:  +0.000  +0.230  -0.103  +0.135

  шаг 2:  ведущий элемент a22 = -1.566
     ведущая строка / -1.566:  -0.000  +1.000  +1.468  +1.358
     строка 3 − (+0.230)·ведущая:  +0.000  +0.000  -0.441  -0.177

  шаг 3:  ведущий элемент a33 = -0.441
     ведущая строка / -0.441:  -0.000  -0.000  +1.000  +0.401

  Треугольный вид:
     +1.000  +2.200  +4.050  +3.700
     -0.000  +1.000  +1.468  +1.358
     -0.000  -0.000  +1.000  +0.401

ОБРАТНЫЙ ХОД
  x3 = +0.401 − (+0.00

In [3]:
# ===== ЗАДАНИЕ 1б — МЕТОД ГАУССА ПРОГРАММОЙ =====
print("=" * 78)
print("ЗАДАНИЕ 1б   Тот же метод Гаусса, но программой (полная точность)")
print("=" * 78)


def gauss(A, b, pivot=False, ndigits=None):
    """Схема единственного деления. pivot=True — с выбором главного элемента."""
    n = len(b)
    M = [row[:] + [b[i]] for i, row in enumerate(A)]
    lead, swaps = [], 0
    rnd = (lambda v: round(v, ndigits)) if ndigits else (lambda v: v)
    for k in range(n):
        if pivot:
            mx = max(range(k, n), key=lambda i: abs(M[i][k]))
            if mx != k:
                M[k], M[mx] = M[mx], M[k]
                swaps += 1
        p = M[k][k]
        lead.append(p)
        M[k] = [rnd(v / p) for v in M[k]]
        for i in range(k + 1, n):
            m = M[i][k]
            M[i] = [rnd(M[i][j] - m * M[k][j]) for j in range(n + 1)]
    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        x[i] = rnd(M[i][n] - sum(M[i][j] * x[j] for j in range(i + 1, n)))
    det = (-1) ** swaps
    for p in lead:
        det *= p
    return x, det, lead


xg, detg, leadg = gauss(A, B)
print("\nа) без выбора главного элемента (та же схема, что и вручную):")
print(f"   x = " + ",  ".join(f"x{i+1} = {v:.9f}" for i, v in enumerate(xg)))
print(f"   det A = {detg:.9f}")
print(f"   невязки: " + "  ".join(f"{v:+.2e}" for v in residuals(A, B, xg)))

xp, detp, leadp = gauss(A, B, pivot=True)
print("\nб) с выбором главного элемента по столбцу:")
print(f"   x = " + ",  ".join(f"x{i+1} = {v:.9f}" for i, v in enumerate(xp)))
print(f"   det A = {detp:.9f}")
print(f"   ведущие элементы: " + ", ".join(f"{p:+.6f}" for p in leadp))
print(f"   невязки: " + "  ".join(f"{v:+.2e}" for v in residuals(A, B, xp)))

print("\n" + "-" * 78)
print("СРАВНЕНИЕ РУЧНОГО И МАШИННОГО СЧЁТА")
print("-" * 78)
xh = RESULTS["Гаусс вручную"]
print(f"\n{'':6}{'вручную (3 знака)':>22}{'программа':>18}{'разность':>14}")
for i in range(3):
    print(f"   x{i+1}{xh[i]:>22.3f}{xg[i]:>18.9f}{abs(xh[i]-xg[i]):>14.2e}")
print(f"\n   det{lead[0]*lead[1]*lead[2]:>22.6f}{detg:>18.9f}"
      f"{abs(lead[0]*lead[1]*lead[2]-detg):>14.2e}")

print("""
Ручной счёт разошёлся с точным решением в третьем знаке, хотя невязки
получились очень маленькими. Это важный момент: МАЛАЯ НЕВЯЗКА НЕ ОЗНАЧАЕТ
МАЛОЙ ОШИБКИ РЕШЕНИЯ. Невязка показывает лишь, насколько хорошо найденный
вектор удовлетворяет уравнениям, а не насколько он близок к истинному.

Первый ведущий элемент здесь равен 0,20 — самый маленький в столбце.
Деление на него в первом же шаге увеличивает остальные коэффициенты
(строка стала 1,000 2,200 4,050 3,700), и последующие округления бьют
по точности сильнее. Выбор главного элемента этого избегает: там первым
ведущим стал бы 0,58.
""")
RESULTS["Гаусс программа"] = xg


ЗАДАНИЕ 1б   Тот же метод Гаусса, но программой (полная точность)

а) без выбора главного элемента (та же схема, что и вручную):
   x = x1 = 0.382976563,  x2 = 0.766417375,  x3 = 0.402692645
   det A = 0.137857000
   невязки: +1.11e-16  +1.80e-16  +5.55e-17

б) с выбором главного элемента по столбцу:
   x = x1 = 0.382976563,  x2 = 0.766417375,  x3 = 0.402692645
   det A = 0.137857000
   ведущие элементы: +0.580000, +0.540000, -0.440156
   невязки: +0.00e+00  +6.94e-18  +5.55e-17

------------------------------------------------------------------------------
СРАВНЕНИЕ РУЧНОГО И МАШИННОГО СЧЁТА
------------------------------------------------------------------------------

           вручную (3 знака)         программа      разность
   x1                 0.384       0.382976563      1.02e-03
   x2                 0.769       0.766417375      2.58e-03
   x3                 0.401       0.402692645      1.69e-03

   det              0.138121       0.137857000      2.64e-04

Ручной счёт разо

In [4]:
# ===== ЗАДАНИЕ 2 — МЕТОД ПРОСТОЙ ИТЕРАЦИИ, ε = 1e-4 =====
print("=" * 78)
print("ЗАДАНИЕ 2   Метод простой итерации, ε = 1e-4")
print("=" * 78)

EPS = 1e-4
Am, Bm = np.array(A), np.array(B)

print("\nПРОВЕРКА ДИАГОНАЛЬНОГО ПРЕОБЛАДАНИЯ в исходном порядке строк:")
for i in range(3):
    s = sum(abs(Am[i, j]) for j in range(3) if j != i)
    print(f"   строка {i+1}:  |{Am[i,i]:+.2f}|  против суммы остальных {s:.2f}"
          f"   →  {'выполнено' if abs(Am[i,i]) > s else 'НЕ выполнено'}")

print("""
Ни в одной строке преобладания нет — в таком виде итерации разойдутся.
Переставим строки так, чтобы на диагонали оказались наибольшие элементы:
уравнение 2 наверх (0,58 в первом столбце), затем уравнение 3 (0,34 во
втором), затем уравнение 1 (0,81 в третьем). Перестановка уравнений
решение не меняет.
""")
P = [1, 2, 0]
A2, B2 = Am[P], Bm[P]
show_system(A2.tolist(), B2.tolist(), "Переставленная система:")
print()
for i in range(3):
    s = sum(abs(A2[i, j]) for j in range(3) if j != i)
    print(f"   строка {i+1}:  |{A2[i,i]:+.2f}| > {s:.2f}   →  выполнено")

C = np.zeros((3, 3))
d = np.zeros(3)
for i in range(3):
    d[i] = B2[i] / A2[i, i]
    for j in range(3):
        if i != j:
            C[i, j] = -A2[i, j] / A2[i, i]

print("\nПриведение к виду x = C·x + d:")
for i in range(3):
    print(f"   x{i+1} = " + " ".join(f"{C[i,j]:+.6f}·x{j+1}" for j in range(3) if j != i)
          + f"  {d[i]:+.6f}")

print("\nНОРМЫ МАТРИЦЫ C в разных метриках:")
for label, p in [("∞  (максимум сумм по строкам)", np.inf),
                 ("1  (максимум сумм по столбцам)", 1),
                 ("E  (евклидова)", "fro")]:
    v = np.linalg.norm(C, p)
    print(f"   ||C||_{label:<32} = {v:.6f}   {'< 1 ✓' if v < 1 else '> 1 ✗ не годится'}")
q = np.linalg.norm(C, np.inf)
print(f"""
   Не всякая метрика подходит: в кубической норме ||C||₁ > 1 и оценка
   неприменима, а в норме ∞ имеем q = {q:.4f} < 1 — её и берём.
   Множитель оценочной формулы (3.15):  q/(1−q) = {q/(1-q):.4f}
""")

print("-" * 78)
print("а) С ОЦЕНКОЙ ПОГРЕШНОСТИ ПО ФОРМУЛЕ (3.15):  ‖x*−xᵏ‖ ≤ q/(1−q)·‖xᵏ−xᵏ⁻¹‖")
print("-" * 78)
xe = RESULTS["эталон"]
print(f"\n{'k':>3}{'x₁':>13}{'x₂':>13}{'x₃':>13}{'‖Δ‖∞':>11}{'оценка':>11}{'факт.':>11}")
prev, cur, k = np.zeros(3), d.copy(), 0
k_strict = None
while k <= 20:
    diff = np.linalg.norm(cur - prev, np.inf)
    est = q / (1 - q) * diff
    err = np.linalg.norm(cur - xe, np.inf)
    print(f"{k:>3}{cur[0]:>13.7f}{cur[1]:>13.7f}{cur[2]:>13.7f}"
          f"{diff:>11.2e}{est:>11.2e}{err:>11.2e}")
    if k > 0 and est < EPS:
        k_strict = k
        break
    prev, cur = cur.copy(), C @ cur + d
    k += 1
x_it = cur.copy()
print(f"\n   оценка {q/(1-q)*np.linalg.norm(cur-prev, np.inf):.2e} < {EPS:g}"
      f"  — критерий выполнен на шаге k = {k_strict}")
print(f"   x = " + ",  ".join(f"{v:.7f}" for v in x_it))

print("\n" + "-" * 78)
print("б) ПО ЭМПИРИЧЕСКОМУ КРИТЕРИЮ:  ‖xᵏ − xᵏ⁻¹‖ < ε")
print("-" * 78)
prev, cur, k = np.zeros(3), d.copy(), 0
while True:
    if k > 0 and np.linalg.norm(cur - prev, np.inf) < EPS:
        break
    prev, cur = cur.copy(), C @ cur + d
    k += 1
err_emp = np.linalg.norm(cur - xe, np.inf)
print(f"\n   критерий сработал на шаге k = {k}")
print(f"   x = " + ",  ".join(f"{v:.7f}" for v in cur))
print(f"   фактическая ошибка {err_emp:.2e}")
print(f"""
   Эмпирический критерий останавливает счёт на шаг РАНЬШЕ строгого
   ({k} против {k_strict}), и фактическая ошибка при этом всё равно
   укладывается в ε. Так бывает часто, но гарантии нет: разность
   соседних приближений меньше настоящей ошибки в q/(1−q) = {q/(1-q):.2f} раза.
   Строгая формула (3.15) закладывает этот запас, эмпирическая — нет.
""")
RESULTS["итерации (3.15)"] = x_it
RESULTS["итерации (эмпир.)"] = cur


ЗАДАНИЕ 2   Метод простой итерации, ε = 1e-4

ПРОВЕРКА ДИАГОНАЛЬНОГО ПРЕОБЛАДАНИЯ в исходном порядке строк:
   строка 1:  |+0.20|  против суммы остальных 1.25   →  НЕ выполнено
   строка 2:  |-0.29|  против суммы остальных 0.63   →  НЕ выполнено
   строка 3:  |+0.10|  против суммы остальных 0.39   →  НЕ выполнено

Ни в одной строке преобладания нет — в таком виде итерации разойдутся.
Переставим строки так, чтобы на диагонали оказались наибольшие элементы:
уравнение 2 наверх (0,58 в первом столбце), затем уравнение 3 (0,34 во
втором), затем уравнение 1 (0,81 в третьем). Перестановка уравнений
решение не меняет.

Переставленная система:
   +0.58·x₁ -0.29·x₂ +0.05·x₃ = +0.02
   +0.05·x₁ +0.34·x₂ +0.10·x₃ = +0.32
   +0.20·x₁ +0.44·x₂ +0.81·x₃ = +0.74

   строка 1:  |+0.58| > 0.34   →  выполнено
   строка 2:  |+0.34| > 0.15   →  выполнено
   строка 3:  |+0.81| > 0.64   →  выполнено

Приведение к виду x = C·x + d:
   x1 = +0.500000·x2 -0.086207·x3  +0.034483
   x2 = -0.147059·x1 -0.294118·x3

In [5]:
# ===== ЗАДАНИЕ 3 — ИНСТРУМЕНТАЛЬНОЕ СРЕДСТВО И СОПОСТАВЛЕНИЕ =====
print("=" * 78)
print("ЗАДАНИЕ 3   Решение в инструментальном средстве и сравнение")
print("=" * 78)

Am, Bm = np.array(A), np.array(B)
print("\nВ задании назван инструментальный пакет; здесь эту роль играют")
print("NumPy и SciPy — стандартные библиотеки линейной алгебры.\n")

x_np = np.linalg.solve(Am, Bm)
print(f"   numpy.linalg.solve (LU-разложение):")
print(f"      x = " + ",  ".join(f"{v:.12f}" for v in x_np))

from scipy.linalg import lu, solve
x_sp = solve(Am, Bm)
Pm, L, U = lu(Am)
print(f"\n   scipy.linalg.solve:")
print(f"      x = " + ",  ".join(f"{v:.12f}" for v in x_sp))
print(f"      расхождение с numpy: {np.max(np.abs(x_np - x_sp)):.2e}")

print(f"\n   LU-разложение с перестановкой (scipy.linalg.lu):")
print("      U (верхняя треугольная) — её диагональ и есть ведущие элементы:")
for i in range(3):
    print("         " + "  ".join(f"{U[i,j]:+.6f}" for j in range(3)))
print(f"      диагональ U: " + ", ".join(f"{U[i,i]:+.6f}" for i in range(3)))
print(f"      det A = ±∏(диагональ U) = {np.linalg.det(Am):+.9f}")

X = x_np
print("\n" + "=" * 78)
print("СОПОСТАВЛЕНИЕ ВСЕХ РЕЗУЛЬТАТОВ")
print("=" * 78)
print(f"\n{'способ':<30}{'x₁':>12}{'x₂':>12}{'x₃':>12}{'‖откл.‖∞':>13}")
print("-" * 78)
order = ["Гаусс вручную", "Гаусс программа",
         "итерации (3.15)", "итерации (эмпир.)"]
for name in order:
    v = np.array(RESULTS[name])
    print(f"{name:<30}" + "".join(f"{v[i]:>12.7f}" for i in range(3))
          + f"{np.max(np.abs(v - X)):>13.2e}")
print(f"{'numpy.linalg.solve':<30}" + "".join(f"{X[i]:>12.7f}" for i in range(3))
      + f"{0.0:>13.2e}")

print(f"\n{'способ':<30}{'r₁':>14}{'r₂':>14}{'r₃':>14}")
print("-" * 78)
for name in order:
    r = residuals(A, B, list(RESULTS[name]))
    print(f"{name:<30}" + "".join(f"{v:>14.2e}" for v in r))
r = residuals(A, B, list(X))
print(f"{'numpy.linalg.solve':<30}" + "".join(f"{v:>14.2e}" for v in r))

print(f"""
{'-' * 78}
КОММЕНТАРИЙ

1. Все способы дали одно и то же решение
      x₁ ≈ {X[0]:.6f},   x₂ ≈ {X[1]:.6f},   x₃ ≈ {X[2]:.6f}
   Расхождения не выходят за пределы точности, заявленной для каждого
   способа, так что противоречий между заданиями 1, 2 и 3 нет.

2. Ручной счёт с тремя знаками отклонился примерно на 2,6·10⁻³ — при том
   что невязки у него порядка 10⁻⁴. Округление до третьего знака в
   промежуточных выкладках стоит примерно одного верного знака в ответе.
   Определитель, собранный из ведущих элементов, тоже разошёлся с точным
   в четвёртом знаке по той же причине.

3. Прямой метод (Гаусс) даёт решение за конечное число действий — 
   для системы 3×3 их порядка n³/3 ≈ 9. Итерационный требует
   больше десяти проходов, но каждый проход дешевле, и с ростом
   размерности это соотношение переворачивается: для больших разреженных
   систем итерации оказываются выгоднее.

4. Итерации применимы не к любой записи системы. В исходном порядке строк
   диагонального преобладания не было ни в одной строке, и процесс
   разошёлся бы. Понадобилась перестановка уравнений — она не меняет
   решения, но делает ||C||∞ = {np.linalg.norm(np.array([[0,0.5,-0.0862069],[-0.14705882,0,-0.29411765],[-0.24691358,-0.54320988,0]]), np.inf):.3f} < 1 и обеспечивает сходимость.

5. Библиотечный solve использует то же LU-разложение, что и метод Гаусса,
   но с выбором главного элемента и в двойной точности. Диагональ матрицы
   U — это в точности ведущие элементы из задания 1.

   ИТОГОВЫЙ ОТВЕТ:
      x₁ = {X[0]:.6f}
      x₂ = {X[1]:.6f}
      x₃ = {X[2]:.6f}
      det A = {np.linalg.det(Am):.6f}
""")


ЗАДАНИЕ 3   Решение в инструментальном средстве и сравнение

В задании назван инструментальный пакет; здесь эту роль играют
NumPy и SciPy — стандартные библиотеки линейной алгебры.

   numpy.linalg.solve (LU-разложение):
      x = 0.382976562670,  0.766417374526,  0.402692645277

   scipy.linalg.solve:
      x = 0.382976562670,  0.766417374526,  0.402692645277
      расхождение с numpy: 0.00e+00

   LU-разложение с перестановкой (scipy.linalg.lu):
      U (верхняя треугольная) — её диагональ и есть ведущие элементы:
         +0.580000  -0.290000  +0.050000
         +0.000000  +0.540000  +0.792759
         +0.000000  +0.000000  -0.440156
      диагональ U: +0.580000, +0.540000, -0.440156
      det A = ±∏(диагональ U) = +0.137857000

СОПОСТАВЛЕНИЕ ВСЕХ РЕЗУЛЬТАТОВ

способ                                  x₁          x₂          x₃     ‖откл.‖∞
------------------------------------------------------------------------------
Гаусс вручную                    0.3840000   0.7690000   0.4010000 